<a href="https://colab.research.google.com/github/prabhnoor14/ds2002-fa26/blob/main/notebooks/04-json-apis%20/2026_10_02_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [2]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [3]:
import pandas as pd
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])
print(lines)

assert len(lines) == 4
assert lines['qty'].sum() == 7
print('Q1 checks passed.')

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-07
Q1 checks passed.


explaination: record_path='lines' makes each line item its own row, and meta copies the order number and vendor_id onto each row. The asserts check we got 4 rows and the qty adds up to 7.

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [5]:
# TODO
vendors = pd.json_normalize(vendors_json)
joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)
print(joined)

unmatched = joined[joined['_merge'] == 'left_only']
print()
print('Unmatched vendor:', list(unmatched['vendor_id'].unique()))
print('Units attached to it:', unmatched['qty'].sum())

           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-07           NaN  NaN  left_only

Unmatched vendor: ['V-07']
Units attached to it: 3


explaination: Left join so no line items get dropped, and indicator=True adds a _merge column showing which rows matched. V-07 shows up as left_only with 3 units because it's not in the vendor list.

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [6]:
# TODO
# Label the unmatched vendor 'Unknown vendor' instead of leaving the name as NaN because:
# Labeling it keeps the total at 7 and makes the data problem visible.
joined['name'] = joined['name'].fillna('Unknown vendor')

qty_by_vendor = joined.groupby('name')['qty'].sum()
print(qty_by_vendor)
print('Total:', qty_by_vendor.sum())

name
Cav Merch         1
Hoos Burgers      3
Unknown vendor    3
Name: qty, dtype: int64
Total: 7


explaination: I filled the missing name with 'Unknown vendor' so groupby doesn't drop it. That way the total stays 7.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [7]:
# TODO
qty_by_zone = joined.groupby('zone')['qty'].sum()
print(qty_by_zone)
print('Top zone:', qty_by_zone.idxmax())
print('Zone total:', qty_by_zone.sum())

# The zone total is 4, not 7. The missing 3 units belong to V-07, which has no zone
# groupby dropped those rows.
# dropna=False
print()
print(joined.groupby('zone', dropna=False)['qty'].sum())

zone
A    4
Name: qty, dtype: int64
Top zone: A
Zone total: 4

zone
A      4
NaN    3
Name: qty, dtype: int64


explaination: Grouped by zone and summed qty. Zone A has 4, and the other 3 are V-07's units, which have no zone so they get dropped.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [8]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
has_lines = [o for o in ragged_json if 'lines' in o]
no_lines  = [o['order'] for o in ragged_json if 'lines' not in o]

ragged_lines = pd.json_normalize(has_lines, record_path='lines', meta=['order', 'vendor_id'])


ragged_lines['qty'] = ragged_lines['qty'].astype('Int64')

print(ragged_lines)
print()
print('Orders with no lines:', no_lines)
print('Rows with missing qty:', ragged_lines['qty'].isna().sum())
print('Rows with qty of zero:', (ragged_lines['qty'] == 0).sum())

    item   qty order vendor_id
0   Soda     2     4      V-01
1  Fries  <NA>     6      V-01

Orders with no lines: [5]
Rows with missing qty: 1
Rows with qty of zero: 0


explaination: Order 5 has no lines so json_normalize would crash on it, so I skip it and print it out instead. Changing qty to Int64 keeps the missing one as NA instead of making it 0.

### Q6 — Validate

In [9]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them? Zero means nothing was sold. Missing means we don't know. If you fill missing with 0, totals and averages come out too low and you can't tell the record was bad anymore.

a) Each quantity gets its own row, so you can just group and sum. Total units per vendor or per item is one groupby. With one row per order the quantities are stuck inside a list and you'd have to loop through them first.

b) Zero means nothing was sold. Missing means we don't know. If you fill missing with 0, totals and averages come out too low and you can't tell the record was bad anymore.

